# Async & Batch Evaluations

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/async-batch-eval)

Submit fire-and-forget async evaluations, poll for results, and run a batch in parallel using the Evaluator SDK with `ThreadPoolExecutor`.

## Install

Start with a fresh Python 3.12 runtime. Set the named keys in your environment or Colab Secrets, or enter them at the hidden prompts. You need quota for eight hosted evaluations: four synchronous and four asynchronous. If a job stops without a verdict, check its stored `error_message` before submitting it again. A rate-limit refusal is not a failing grade.

In [ ]:
%pip install agent-learning-kit==0.3.7

In [ ]:
import os
from getpass import getpass

def load_secret(name):
    if os.environ.get(name):
        return
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except (ImportError, KeyError, RuntimeError):
        value = None
    except Exception as exc:
        if type(exc).__name__ not in {"SecretNotFoundError", "NotebookAccessError"}:
            raise
        value = None
    value = value or getpass(f"{name}: ")
    if not value:
        raise ValueError(f"{name} is required")
    os.environ[name] = value

for name in ['FI_API_KEY', 'FI_SECRET_KEY']:
    load_secret(name)

## Run a synchronous eval as a baseline

In [ ]:
from fi.evals import Evaluator

evaluator = Evaluator()
cases = [
    {"context": "Standard shipping takes 3 to 5 days.", "output": "Standard shipping takes 3 to 5 days."},
    {"context": "Final-sale items cannot be refunded.", "output": "Final-sale items qualify for a full refund."},
    {"context": "Prepaid return labels are available on the Orders page.", "output": "Find your prepaid return label on the Orders page."},
]

def score(case, *, background=False):
    batch = evaluator.evaluate(
        eval_templates="groundedness", inputs=case, model_name="turing_small", is_async=background
    )
    return batch.eval_results[0]

baseline = score(cases[0])
print(f"Baseline: {baseline.output}")
print(baseline.reason)

## Submit an async evaluation

In [ ]:
single_id = score(cases[0], background=True).eval_id
print(f"Submitted async eval: {single_id}")

## Poll for the stored result

In [ ]:
import time

def read_if_ready(eval_id):
    job = evaluator.get_eval_result(eval_id)["result"]
    if job["eval_status"] not in {"completed", "failed"}:
        return None
    detail = job["result"]
    if not isinstance(detail, dict) or detail.get("value") not in {"Passed", "Failed"}:
        raise RuntimeError(f"Evaluation {eval_id} stopped without a groundedness verdict")
    return detail

for _ in range(60):
    detail = read_if_ready(single_id)
    if detail is not None:
        print(f"{detail['name']}: {detail['value']}")
        break
    time.sleep(2)
else:
    raise TimeoutError(f"Still waiting for {single_id}")

## Score independent answers in parallel

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

sync_ids = {}
with ThreadPoolExecutor(max_workers=3) as pool:
    pending = {pool.submit(score, case): i for i, case in enumerate(cases)}
    for future in as_completed(pending):
        i = pending[future]
        verdict = future.result()
        sync_ids[i] = verdict.eval_id
        print(f"Case {i + 1}: {verdict.output}")

## Combine async submission with batch polling

In [ ]:
import json
from pathlib import Path

with ThreadPoolExecutor(max_workers=3) as pool:
    pending = {pool.submit(score, case, background=True): i for i, case in enumerate(cases)}
    async_ids = {pending[f]: f.result().eval_id for f in as_completed(pending)}

Path("eval-ids.json").write_text(json.dumps({
    "baseline": baseline.eval_id, "single": single_id, "sync": sync_ids, "async": async_ids,
}, indent=2))

remaining = dict(async_ids)
for _ in range(60):
    for i, eval_id in list(remaining.items()):
        detail = read_if_ready(eval_id)
        if detail is not None:
            print(f"Case {i + 1}: {detail['value']}")
            del remaining[i]
    if not remaining:
        break
    time.sleep(2)
else:
    raise TimeoutError(f"Resume polling these evaluation IDs: {list(remaining.values())}")
print(f"Finished {len(async_ids)} async evaluations")